In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# ============================================================
# A5 REPLAY EXPORT FOR HUGGING FACE
# Corrected version
#
# Exports one replay page from observable_inputs.npz into
# ordinary PNG / CSV / JSON files suitable for the HF-04.3
# mandatory replay gate.
#
# IMPORTANT:
#   - image-like arrays -> PNG
#   - depth -> sampled at replay y/x coordinates -> CSV
#   - x/y -> 5000-row CSV vectors WITH HEADERS
#   - expected replay outputs -> 5000-row CSV vectors
#   - full depth matrix is NOT exported
# ============================================================

from pathlib import Path
import json
import hashlib

import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# EDIT ONLY THESE PATHS
# ------------------------------------------------------------

REPLAY_DIR = Path(
    "/content/drive/MyDrive/MusicAnnotationProject/"
    "a5_v1_freeze/frozen/"
    "A5-v1-print-p0980-margin0960-run2/"
    "826035ea9745644a0b21b10ec1fc3d1b08670f65f2a5537e1116f4428f1a9f59/replay/Page003"
)

OUTPUT_DIR = Path("/content/ardross_hf_replay_export")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

INPUT_NPZ = REPLAY_DIR / "observable_inputs.npz"

EXPECTED_PROBABILITY = REPLAY_DIR / "expected_probability.npy"
EXPECTED_PROVISIONAL = REPLAY_DIR / "expected_provisional.npy"
EXPECTED_FINAL = REPLAY_DIR / "expected_final.npy"
EXPECTED_A5 = REPLAY_DIR / "expected_a5.png"


required_files = [
    INPUT_NPZ,
    EXPECTED_PROBABILITY,
    EXPECTED_PROVISIONAL,
    EXPECTED_FINAL,
    EXPECTED_A5,
]

missing = [
    str(p)
    for p in required_files
    if not p.exists()
]

if missing:
    raise RuntimeError(
        "Missing required replay files:\n"
        + "\n".join(missing)
    )


# ------------------------------------------------------------
# SEMANTIC ARRAY HASH
# ------------------------------------------------------------

def array_hash(a):
    a = np.ascontiguousarray(a)

    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())

    return h.hexdigest()


# ------------------------------------------------------------
# LOAD OBSERVABLE INPUTS
# ------------------------------------------------------------

with np.load(INPUT_NPZ, allow_pickle=False) as z:

    print("NPZ arrays:")
    for name in z.files:
        a = z[name]
        print(
            f"  {name:12s}",
            "shape=", a.shape,
            "dtype=", a.dtype
        )

    required_arrays = [
        "obs",
        "norm",
        "mask",
        "depth",
        "horizontal",
        "vertical",
        "compact",
        "preserve",
        "remove",
        "ambiguous",
        "edge",
        "a1",
        "y",
        "x",
    ]

    absent = [
        name
        for name in required_arrays
        if name not in z.files
    ]

    if absent:
        raise RuntimeError(
            f"observable_inputs.npz missing arrays: {absent}"
        )

    arrays = {
        name: z[name].copy()
        for name in required_arrays
    }


# ------------------------------------------------------------
# VALIDATE PAGE ARRAYS
# ------------------------------------------------------------

obs = arrays["obs"]

if obs.ndim != 2:
    raise RuntimeError(
        f"obs must be 2-D; got {obs.shape}"
    )

H, W = obs.shape

PAGE_ARRAYS = [
    "norm",
    "mask",
    "depth",
    "horizontal",
    "vertical",
    "compact",
    "preserve",
    "remove",
    "ambiguous",
    "edge",
    "a1",
]

for name in PAGE_ARRAYS:

    a = arrays[name]

    if a.shape != (H, W):
        raise RuntimeError(
            f"{name} shape mismatch: "
            f"{a.shape} != obs {(H, W)}"
        )

    if not np.isfinite(a).all():
        raise RuntimeError(
            f"{name} contains non-finite values"
        )


# ------------------------------------------------------------
# VALIDATE REPLAY COORDINATES
# ------------------------------------------------------------

y = np.asarray(arrays["y"]).reshape(-1)
x = np.asarray(arrays["x"]).reshape(-1)

if len(y) != len(x):
    raise RuntimeError(
        f"x/y lengths differ: x={len(x)}, y={len(y)}"
    )

if len(x) == 0:
    raise RuntimeError("Replay coordinate set is empty")

if not np.issubdtype(x.dtype, np.integer):
    if not np.all(x == np.round(x)):
        raise RuntimeError("x contains non-integer coordinates")

if not np.issubdtype(y.dtype, np.integer):
    if not np.all(y == np.round(y)):
        raise RuntimeError("y contains non-integer coordinates")

x = x.astype(np.int64)
y = y.astype(np.int64)

if (
    (x < 0).any()
    or (x >= W).any()
    or (y < 0).any()
    or (y >= H).any()
):
    raise RuntimeError(
        "Replay contains out-of-bounds coordinates"
    )

N = len(x)

print()
print("Replay page shape:", (H, W))
print("Replay coordinates:", N)


# ------------------------------------------------------------
# LOAD EXPECTED REPLAY OUTPUTS
# ------------------------------------------------------------

expected_probability = np.asarray(
    np.load(EXPECTED_PROBABILITY, allow_pickle=True)
).reshape(-1)

expected_provisional = np.asarray(
    np.load(EXPECTED_PROVISIONAL, allow_pickle=True)
).reshape(-1)

expected_final = np.asarray(
    np.load(EXPECTED_FINAL, allow_pickle=True)
).reshape(-1)


# ------------------------------------------------------------
# CRITICAL ROW-COUNT VALIDATION
# ------------------------------------------------------------

counts = {
    "x": len(x),
    "y": len(y),
    "expected_probability": len(expected_probability),
    "expected_provisional": len(expected_provisional),
    "expected_final": len(expected_final),
}

print()
print("Replay vector counts:")
for name, count in counts.items():
    print(f"  {name:24s} {count}")

if len(set(counts.values())) != 1:
    raise RuntimeError(
        "SOURCE REPLAY VECTORS HAVE INCONSISTENT "
        f"ROW COUNTS: {counts}"
    )

if N != 5000:
    raise RuntimeError(
        f"Expected 5000 replay coordinates; found {N}"
    )


# ------------------------------------------------------------
# EXPORT IMAGE-LIKE CHANNELS AS LOSSLESS PNG
# ------------------------------------------------------------

def save_gray_png(name, a):

    a = np.asarray(a)

    if a.dtype == bool:
        out = a.astype(np.uint8) * 255

    elif a.dtype == np.uint8:
        out = a

    elif np.array_equal(a, a.astype(bool)):
        out = a.astype(np.uint8) * 255

    else:
        raise RuntimeError(
            f"{name} is not suitable for lossless "
            f"PNG export: dtype={a.dtype}, "
            f"range=({a.min()}, {a.max()})"
        )

    path = OUTPUT_DIR / f"{name}.png"

    if not cv2.imwrite(str(path), out):
        raise RuntimeError(
            f"Failed to write {path}"
        )

    return path


# obs / norm / A1 retain their byte values
save_gray_png("obs", arrays["obs"])
save_gray_png("norm", arrays["norm"])
save_gray_png("a1", arrays["a1"])

# binary channels
for name in [
    "mask",
    "horizontal",
    "vertical",
    "compact",
    "preserve",
    "remove",
    "ambiguous",
    "edge",
]:
    save_gray_png(name, arrays[name])


# ------------------------------------------------------------
# DEPTH:
# EXPORT ONLY THE 5000 VALUES AT REPLAY COORDINATES
# ------------------------------------------------------------

depth = np.asarray(
    arrays["depth"],
    dtype=np.float32,
)

depth_at_replay = depth[y, x]

if depth_at_replay.shape != (N,):
    raise RuntimeError(
        "Sampled depth has unexpected shape: "
        f"{depth_at_replay.shape}"
    )

pd.DataFrame({
    "depth": depth_at_replay
}).to_csv(
    OUTPUT_DIR / "depth.csv",
    index=False,
)


# ------------------------------------------------------------
# EXPORT COORDINATES
# EXPLICIT HEADERS ARE IMPORTANT
# ------------------------------------------------------------

pd.DataFrame({
    "x": x
}).to_csv(
    OUTPUT_DIR / "x.csv",
    index=False,
)

pd.DataFrame({
    "y": y
}).to_csv(
    OUTPUT_DIR / "y.csv",
    index=False,
)


# ------------------------------------------------------------
# EXPORT EXPECTED REPLAY OUTPUTS
# ------------------------------------------------------------

pd.DataFrame({
    "expected_probability": expected_probability
}).to_csv(
    OUTPUT_DIR / "expected_probability.csv",
    index=False,
)

pd.DataFrame({
    "expected_provisional": expected_provisional
}).to_csv(
    OUTPUT_DIR / "expected_provisional.csv",
    index=False,
)

pd.DataFrame({
    "expected_final": expected_final
}).to_csv(
    OUTPUT_DIR / "expected_final.csv",
    index=False,
)

expected_a5 = cv2.imread(
    str(EXPECTED_A5),
    cv2.IMREAD_GRAYSCALE,
)

if expected_a5 is None:
    raise RuntimeError(
        "Could not read expected_a5.png"
    )

if expected_a5.shape != (H, W):
    raise RuntimeError(
        "expected_a5 shape mismatch: "
        f"{expected_a5.shape} != {(H, W)}"
    )

cv2.imwrite(
    str(OUTPUT_DIR / "expected_a5.png"),
    expected_a5,
)


# ------------------------------------------------------------
# CREATE SEMANTIC SOURCE MANIFEST
# ------------------------------------------------------------

manifest = {}

for name, a in arrays.items():

    a = np.asarray(a)

    manifest[name] = {
        "shape": list(a.shape),
        "dtype": str(a.dtype),
        "sha256": array_hash(a),
    }

manifest["expected_probability"] = {
    "shape": list(expected_probability.shape),
    "dtype": str(expected_probability.dtype),
    "sha256": array_hash(expected_probability),
}

manifest["expected_provisional"] = {
    "shape": list(expected_provisional.shape),
    "dtype": str(expected_provisional.dtype),
    "sha256": array_hash(expected_provisional),
}

manifest["expected_final"] = {
    "shape": list(expected_final.shape),
    "dtype": str(expected_final.dtype),
    "sha256": array_hash(expected_final),
}

manifest["expected_a5"] = {
    "shape": list(expected_a5.shape),
    "dtype": str(expected_a5.dtype),
    "sha256": array_hash(expected_a5),
}

manifest["export"] = {
    "page_shape": [H, W],
    "replay_rows": N,
    "depth_representation":
        "depth[y,x] at frozen replay coordinates",
    "coordinate_csv_headers": True,
}

(
    OUTPUT_DIR / "array_manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    )
)


# ------------------------------------------------------------
# VALIDATE THE EXPORTED CSVs BY READING THEM BACK
# ------------------------------------------------------------

vector_files = [
    "x.csv",
    "y.csv",
    "depth.csv",
    "expected_probability.csv",
    "expected_provisional.csv",
    "expected_final.csv",
]

print()
print("Export validation:")

for filename in vector_files:

    path = OUTPUT_DIR / filename
    df = pd.read_csv(path)

    print(
        f"  {filename:28s}",
        f"rows={len(df):5d}",
        f"columns={list(df.columns)}"
    )

    if len(df) != N:
        raise RuntimeError(
            f"{filename}: exported row count "
            f"{len(df)} != {N}"
        )

    if len(df.columns) != 1:
        raise RuntimeError(
            f"{filename}: expected one column, "
            f"found {len(df.columns)}"
        )


# ------------------------------------------------------------
# FINAL FILE INVENTORY
# ------------------------------------------------------------

print()
print("EXPORT PASSED")
print("Output:", OUTPUT_DIR)
print()
print("Files:")

for path in sorted(OUTPUT_DIR.iterdir()):
    print(
        f"  {path.name:30s}",
        f"{path.stat().st_size:,} bytes"
    )

print()
print(
    "All six replay CSV vectors contain exactly",
    N,
    "data rows."
)